**Important liaberies**

In [12]:
import numpy as np
import pandas as pd

**Step1: Upload Your Dataset**

In [13]:
df = pd.read_csv('/content/raw_car_dataset.csv')

**Step 2:Inspect the Dataset**

In [14]:
# Display the first 10 rows of the dataset
print("First 10 Rows")
print(df.head(10))

First 10 Rows
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [15]:
# Display the number of rows and columns
print("\nShape")
print(df.shape)


Shape
(145, 6)


In [16]:
# Display dataset structure, data types, and non-null counts
print("\nDataset Info")
print(df.info())


Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


In [17]:
# Display the number of missing values in each column
print("\nMissing Values")
print(df.isnull().sum())


Missing Values
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [18]:
# Display the frequency count of each category in Location
print("\nLocation Counts")
print(df['Location'].value_counts(dropna=False))


Location Counts
Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


**Step 3: Clean the Price Column**

In [19]:
# Remove $ and commas

df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

# Convert to numeric
df['Price'] = pd.to_numeric(df['Price'])

print("Price Data Type:")
print(df['Price'].dtype)

print("\nPrice Skewness:")
print(df['Price'].skew())

Price Data Type:
float64

Price Skewness:
7.871113660850301


**step 4: Fix Location Errors**

In [20]:

# Standardize text
df['Location'] = df['Location'].str.strip().str.title()

# Correct errors
df['Location'] = df['Location'].replace({
    'Subrb': 'Suburb',
    '??': np.nan
})

print(df['Location'].value_counts(dropna=False))

Location
City      59
Suburb    53
Rural     21
NaN       12
Name: count, dtype: int64


**Step 5:Fill Missing Values**

In [21]:
# Odometer -> Median
df['Odometer_km'] = df['Odometer_km'].fillna(
    df['Odometer_km'].median()
)

# Doors -> Mode
df['Doors'] = df['Doors'].fillna(
    df['Doors'].mode()[0]
)

# Accidents -> Mode
df['Accidents'] = df['Accidents'].fillna(
    df['Accidents'].mode()[0]
)

# Location -> Mode
df['Location'] = df['Location'].fillna(
    df['Location'].mode()[0]
)

print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


**step 6:Remove Duplicates**

In [22]:
before = df.shape[0]

df = df.drop_duplicates()

after = df.shape[0]

print("Rows Before:", before)
print("Rows After:", after)
print("Removed:", before - after)

Rows Before: 145
Rows After: 140
Removed: 5


**step 7:Handle Outliers Using IQR Capping**

In [23]:
for col in ['Price', 'Odometer_km']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(f"\n{col}")
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)

    # Cap values
    df[col] = df[col].clip(lower, upper)

print(df[['Price', 'Odometer_km']].describe())


Price
Lower Bound: -2984.625
Upper Bound: 8974.375

Odometer_km
Lower Bound: -6642.25
Upper Bound: 271987.75
             Price    Odometer_km
count   140.000000     140.000000
mean   3175.456250  130945.403571
std    2601.848629   53815.006935
min    1500.000000    5000.000000
25%    1500.000000   97844.000000
50%    1500.000000  128548.000000
75%    4489.750000  167501.500000
max    8974.375000  271987.750000


**step 8:One-Hot Encode Location**

In [24]:
df = pd.get_dummies(
    df,
    columns=['Location'],
    prefix='Location'
)

print(df.columns)

Index(['Price', 'Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_City',
       'Location_Rural', 'Location_Suburb'],
      dtype='object')


**step 9: Feature Engineering**

In [25]:
current_year = 2026

# Car Age
df['CarAge'] = current_year - df['Year']

# Km per Year
df['Km_per_year'] = np.where(
    df['CarAge'] > 0,
    df['Odometer_km'] / df['CarAge'],
    0
)

# Urban Indicator
df['Is_Urban'] = (
    (df['Location_City'] == 1) |
    (df['Location_Suburb'] == 1)
).astype(int)

# Alternative Target
df['LogPrice'] = np.log(df['Price'] + 1)

print(df[['CarAge',
          'Km_per_year',
          'Is_Urban',
          'LogPrice']].head())

   CarAge   Km_per_year  Is_Urban  LogPrice
0      28   4922.500000         1  7.313887
1      10  12854.800000         0  8.336151
2      12   8941.833333         1  8.581482
3      27   5253.259259         1  7.313887
4       4  32137.000000         1  7.313887


**step 10: Scale Continuous Features**

In [26]:
from sklearn.preprocessing import StandardScaler

In [27]:
scaler = StandardScaler()

continuous_cols = [
    'Odometer_km',
    'CarAge',
    'Km_per_year'
]

df[continuous_cols] = scaler.fit_transform(
    df[continuous_cols]
)

print(df[continuous_cols].head())

   Odometer_km    CarAge  Km_per_year
0     0.128390  1.686714    -0.615631
1    -0.044709 -0.794617     0.070446
2    -0.440923 -0.518913    -0.267993
3     0.203135  1.548862    -0.587024
4    -0.044709 -1.621727     1.738196


**step 11: Final Checks**

In [28]:
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nStatistics")
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    int64  
 4   Year             140 non-null    int64  
 5   Location_City    140 non-null    bool   
 6   Location_Rural   140 non-null    bool   
 7   Location_Suburb  140 non-null    bool   
 8   CarAge           140 non-null    float64
 9   Km_per_year      140 non-null    float64
 10  Is_Urban         140 non-null    int64  
 11  LogPrice         140 non-null    float64
dtypes: bool(3), float64(6), int64(3)
memory usage: 11.3 KB
None

Missing Values
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Suburb    0
CarAge 

**Save the Clean Dataset**

In [31]:
# Save the cleaned
df.to_csv(
    '/content/car_l3_clean_ready.csv',
    index=False
)

print("Dataset saved successfully as car_l3_clean_ready.csv")

Dataset saved successfully as car_l3_clean_ready.csv


In [32]:
# Create requirements.txt file
with open('requirements.txt', 'w') as f:
    f.write('pandas==2.3.0\n')
    f.write('numpy==2.3.1\n')
    f.write('scikit-learn==1.7.0\n')

print("requirements.txt created successfully.")

requirements.txt created successfully.
